# 🧠 Axom AI — Fine-tune (Assam assistant: English + Hinglish)

**Run har cell top-to-bottom** (Kaggle: Run All). GPU on karo (Settings → Accelerator → GPU T4).

Data: `assam_train.jsonl` upload karo (12k+ Q&A). ~12k examples pe training ~15-30 min lagega.


## 1. Install Unsloth


In [ ]:
!pip install -q unsloth


## 2. Load base model


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-1B-Instruct',
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)


## 3. Add LoRA adapters


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)


## 4. Load data (upload assam_train.jsonl)


In [ ]:
import glob, json
from datasets import Dataset

matches = (glob.glob('/kaggle/input/**/assam_train.jsonl', recursive=True)
           or glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
           or glob.glob('*.jsonl'))
DATA_PATH = matches[0]
print('Using data file:', DATA_PATH)

rows = []
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print('Total examples:', len(rows))

def to_chat(ex):
    msgs = [
        {'role': 'user', 'content': ex['instruction']},
        {'role': 'assistant', 'content': ex['output']},
    ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False)}

dataset = Dataset.from_list(rows).map(to_chat)
print(dataset[0]['text'][:300])


## 5. Train (batched = fast for large data)


In [ ]:
import torch

FastLanguageModel.for_training(model)
model.train()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4
)

texts = dataset['text']
BATCH = 8
EPOCHS = 2
steps_per_epoch = (len(texts) + BATCH - 1) // BATCH

for epoch in range(EPOCHS):
    total = 0.0; nb = 0
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i+BATCH]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=max_seq_length).to('cuda')
        labels = enc['input_ids'].clone()
        labels[enc['attention_mask'] == 0] = -100
        out = model(input_ids=enc['input_ids'],
                    attention_mask=enc['attention_mask'], labels=labels)
        loss = out.loss
        loss.backward(); optimizer.step(); optimizer.zero_grad()
        total += loss.item(); nb += 1
        if nb % 100 == 0:
            print(f'  epoch {epoch+1} step {nb}/{steps_per_epoch}  loss {loss.item():.4f}')
    print(f'Epoch {epoch+1}/{EPOCHS}  avg loss = {total/nb:.4f}')
print('Training done!')


## 6. Quick test


In [ ]:
FastLanguageModel.for_inference(model)
for q in ['Assam ki rajdhani kya hai?', 'What is Bihu?', 'Kaziranga kis ke liye famous hai?']:
    msgs = [{'role':'user','content': q}]
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inp, max_new_tokens=120, temperature=0.7)
    print('Q:', q)
    print(tokenizer.decode(out[0], skip_special_tokens=True).split('assistant')[-1].strip()[:200])
    print('---')


## 7. Export to GGUF + download


In [ ]:
model.save_pretrained_gguf('axom_model', tokenizer, quantization_method='q4_k_m')
import glob
print('GGUF files:', glob.glob('/kaggle/working/**/*.gguf', recursive=True))


## 8. Apne PC pe Ollama me chalao

GGUF download → us folder me `Modelfile` banao:
```
FROM ./llama-3.2-1b-instruct.Q4_K_M.gguf
```
Phir:
```
ollama create axom-assam -f Modelfile
```
Axom AI `.env`:  `OLLAMA_MODEL=axom-assam`  → Django restart. 🎉
